# Análisis de Noticias de Yahoo Finance

Este notebook realiza un análisis de las noticias principales de Yahoo Finance utilizando pandas.

## 1. Instalación de Dependencias

Primero instalamos las librerías necesarias.

In [ ]:
# Instalar dependencias necesarias
!pip install pandas feedparser requests beautifulsoup4 matplotlib wordcloud textblob -q

## 2. Importar Librerías

In [ ]:
import pandas as pd
import feedparser
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import matplotlib.pyplot as plt
from collections import Counter
import re
from wordcloud import WordCloud
from textblob import TextBlob
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')

print("Librerías importadas correctamente.")

## 3. Obtener Noticias de Yahoo Finance

Utilizamos el feed RSS de Yahoo Finance para obtener las noticias más recientes.

In [ ]:
def get_yahoo_finance_news():
    """
    Obtiene las noticias de Yahoo Finance desde múltiples fuentes RSS.
    """
    # URLs de feeds RSS de Yahoo Finance
    rss_urls = [
        'https://finance.yahoo.com/news/rssindex',
        'https://finance.yahoo.com/rss/topstories',
        'https://finance.yahoo.com/rss/industry?s=technology',
    ]
    
    all_news = []
    
    for url in rss_urls:
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries:
                news_item = {
                    'titulo': entry.get('title', 'Sin título'),
                    'descripcion': entry.get('summary', entry.get('description', '')),
                    'link': entry.get('link', ''),
                    'fecha_publicacion': entry.get('published', ''),
                    'fuente': feed.feed.get('title', 'Yahoo Finance')
                }
                all_news.append(news_item)
        except Exception as e:
            print(f"Error al obtener feed {url}: {e}")
    
    return all_news

# Obtener noticias
print("Obteniendo noticias de Yahoo Finance...")
noticias = get_yahoo_finance_news()
print(f"Se obtuvieron {len(noticias)} noticias.")

## 4. Crear DataFrame con Pandas

In [ ]:
# Crear DataFrame
df_noticias = pd.DataFrame(noticias)

# Limpiar HTML de las descripciones
def limpiar_html(texto):
    if pd.isna(texto):
        return ''
    soup = BeautifulSoup(texto, 'html.parser')
    return soup.get_text(separator=' ').strip()

df_noticias['descripcion_limpia'] = df_noticias['descripcion'].apply(limpiar_html)

# Convertir fechas
def parse_fecha(fecha_str):
    if pd.isna(fecha_str) or fecha_str == '':
        return None
    try:
        return pd.to_datetime(fecha_str)
    except:
        return None

df_noticias['fecha'] = df_noticias['fecha_publicacion'].apply(parse_fecha)

# Eliminar duplicados basados en el título
df_noticias = df_noticias.drop_duplicates(subset=['titulo'])

print(f"DataFrame creado con {len(df_noticias)} noticias únicas.")
df_noticias.head()

## 5. Información General del Dataset

In [ ]:
print("=" * 60)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 60)
print(f"\nNúmero total de noticias: {len(df_noticias)}")
print(f"\nColumnas disponibles: {list(df_noticias.columns)}")
print(f"\nTipos de datos:")
print(df_noticias.dtypes)
print(f"\nValores nulos por columna:")
print(df_noticias.isnull().sum())

## 6. Análisis de Sentimiento

In [ ]:
def analizar_sentimiento(texto):
    """
    Analiza el sentimiento del texto usando TextBlob.
    Retorna la polaridad (-1 a 1) y la subjetividad (0 a 1).
    """
    if pd.isna(texto) or texto == '':
        return 0, 0
    try:
        blob = TextBlob(str(texto))
        return blob.sentiment.polarity, blob.sentiment.subjectivity
    except:
        return 0, 0

# Aplicar análisis de sentimiento
df_noticias['sentimiento'] = df_noticias['titulo'].apply(lambda x: analizar_sentimiento(x)[0])
df_noticias['subjetividad'] = df_noticias['titulo'].apply(lambda x: analizar_sentimiento(x)[1])

# Clasificar sentimiento
def clasificar_sentimiento(polaridad):
    if polaridad > 0.1:
        return 'Positivo'
    elif polaridad < -0.1:
        return 'Negativo'
    else:
        return 'Neutral'

df_noticias['clasificacion_sentimiento'] = df_noticias['sentimiento'].apply(clasificar_sentimiento)

print("Análisis de sentimiento completado.")
df_noticias[['titulo', 'sentimiento', 'clasificacion_sentimiento']].head(10)

## 7. Distribución de Sentimientos

In [ ]:
# Contar sentimientos
conteo_sentimientos = df_noticias['clasificacion_sentimiento'].value_counts()

# Crear gráfico de pastel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de pastel
colors = {'Positivo': '#2ecc71', 'Neutral': '#3498db', 'Negativo': '#e74c3c'}
color_list = [colors.get(x, '#95a5a6') for x in conteo_sentimientos.index]

axes[0].pie(conteo_sentimientos.values, labels=conteo_sentimientos.index, 
            autopct='%1.1f%%', colors=color_list, startangle=90)
axes[0].set_title('Distribución de Sentimientos en Noticias', fontsize=12, fontweight='bold')

# Histograma de polaridad
axes[1].hist(df_noticias['sentimiento'], bins=20, color='#3498db', edgecolor='white', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--', label='Neutral')
axes[1].set_xlabel('Polaridad del Sentimiento')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Polaridad', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nResumen de sentimientos:")
print(conteo_sentimientos)

## 8. Análisis de Palabras Clave

In [ ]:
def extraer_palabras(texto):
    """
    Extrae palabras significativas del texto.
    """
    if pd.isna(texto):
        return []
    
    # Palabras a ignorar (stopwords básicas)
    stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
                 'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been',
                 'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
                 'could', 'should', 'may', 'might', 'must', 'shall', 'can', 'this',
                 'that', 'these', 'those', 'i', 'you', 'he', 'she', 'it', 'we', 'they',
                 'what', 'which', 'who', 'whom', 'whose', 'where', 'when', 'why', 'how',
                 'all', 'each', 'every', 'both', 'few', 'more', 'most', 'other', 'some',
                 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too',
                 'very', 's', 't', 'just', 'don', 'now', 'its', 'into', 'over', 'after',
                 'says', 'say', 'said', 'about', 'new', 'out', 'up', 'down'}
    
    # Limpiar y tokenizar
    texto = texto.lower()
    palabras = re.findall(r'\b[a-z]{3,}\b', texto)
    
    return [p for p in palabras if p not in stopwords]

# Extraer todas las palabras
todas_palabras = []
for titulo in df_noticias['titulo']:
    todas_palabras.extend(extraer_palabras(titulo))

# Contar frecuencias
conteo_palabras = Counter(todas_palabras)
top_20_palabras = conteo_palabras.most_common(20)

print("Top 20 palabras más frecuentes en los títulos:")
print("=" * 40)
for palabra, frecuencia in top_20_palabras:
    print(f"{palabra}: {frecuencia}")

## 9. Nube de Palabras

In [ ]:
# Crear texto combinado para la nube de palabras
texto_combinado = ' '.join(todas_palabras)

if texto_combinado.strip():
    # Generar nube de palabras
    wordcloud = WordCloud(
        width=1200,
        height=600,
        background_color='white',
        colormap='viridis',
        max_words=100,
        min_font_size=10
    ).generate(texto_combinado)

    # Mostrar nube de palabras
    plt.figure(figsize=(15, 7))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Nube de Palabras - Noticias de Yahoo Finance', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No hay suficientes palabras para generar la nube de palabras.")

## 10. Análisis de Fuentes

In [ ]:
# Contar noticias por fuente
noticias_por_fuente = df_noticias['fuente'].value_counts()

if len(noticias_por_fuente) > 0:
    plt.figure(figsize=(10, 6))
    noticias_por_fuente.plot(kind='barh', color='#3498db', edgecolor='white')
    plt.xlabel('Número de Noticias')
    plt.ylabel('Fuente')
    plt.title('Noticias por Fuente', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\nDistribución por fuente:")
print(noticias_por_fuente)

## 11. Estadísticas Descriptivas

In [ ]:
# Calcular longitud de títulos y descripciones
df_noticias['longitud_titulo'] = df_noticias['titulo'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)
df_noticias['num_palabras_titulo'] = df_noticias['titulo'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)

print("=" * 60)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 60)

print("\n--- Longitud de Títulos ---")
print(f"Promedio de caracteres: {df_noticias['longitud_titulo'].mean():.1f}")
print(f"Promedio de palabras: {df_noticias['num_palabras_titulo'].mean():.1f}")
print(f"Título más largo: {df_noticias['longitud_titulo'].max()} caracteres")
print(f"Título más corto: {df_noticias['longitud_titulo'].min()} caracteres")

print("\n--- Sentimiento ---")
print(f"Polaridad promedio: {df_noticias['sentimiento'].mean():.3f}")
print(f"Polaridad máxima: {df_noticias['sentimiento'].max():.3f}")
print(f"Polaridad mínima: {df_noticias['sentimiento'].min():.3f}")
print(f"Subjetividad promedio: {df_noticias['subjetividad'].mean():.3f}")

## 12. Noticias Más Positivas y Negativas

In [ ]:
print("=" * 60)
print("TOP 5 NOTICIAS MÁS POSITIVAS")
print("=" * 60)
top_positivas = df_noticias.nlargest(5, 'sentimiento')[['titulo', 'sentimiento']]
for idx, row in top_positivas.iterrows():
    print(f"\n[Polaridad: {row['sentimiento']:.3f}]")
    print(f"  {row['titulo']}")

print("\n")
print("=" * 60)
print("TOP 5 NOTICIAS MÁS NEGATIVAS")
print("=" * 60)
top_negativas = df_noticias.nsmallest(5, 'sentimiento')[['titulo', 'sentimiento']]
for idx, row in top_negativas.iterrows():
    print(f"\n[Polaridad: {row['sentimiento']:.3f}]")
    print(f"  {row['titulo']}")

## 13. Detección de Temas/Categorías

In [ ]:
# Definir categorías y palabras clave asociadas
categorias = {
    'Tecnología': ['tech', 'technology', 'ai', 'artificial', 'intelligence', 'software', 
                   'apple', 'google', 'microsoft', 'amazon', 'meta', 'nvidia', 'chip', 'chips'],
    'Mercados': ['stock', 'stocks', 'market', 'markets', 'dow', 'nasdaq', 'sp500', 
                 'trading', 'investor', 'investors', 'rally', 'bull', 'bear'],
    'Criptomonedas': ['bitcoin', 'crypto', 'cryptocurrency', 'ethereum', 'blockchain', 
                      'btc', 'eth', 'token', 'defi'],
    'Economía': ['economy', 'economic', 'inflation', 'fed', 'federal', 'reserve', 
                 'interest', 'rate', 'rates', 'gdp', 'recession'],
    'Empresas': ['earnings', 'revenue', 'profit', 'ceo', 'company', 'corporate', 
                 'business', 'deal', 'merger', 'acquisition'],
    'Energía': ['oil', 'gas', 'energy', 'solar', 'renewable', 'electric', 'ev', 
                'tesla', 'fuel', 'petroleum']
}

def detectar_categoria(titulo):
    if pd.isna(titulo):
        return 'Otros'
    titulo_lower = titulo.lower()
    for categoria, keywords in categorias.items():
        for keyword in keywords:
            if keyword in titulo_lower:
                return categoria
    return 'Otros'

df_noticias['categoria'] = df_noticias['titulo'].apply(detectar_categoria)

# Visualizar distribución de categorías
conteo_categorias = df_noticias['categoria'].value_counts()

plt.figure(figsize=(10, 6))
colors = plt.cm.Set3(range(len(conteo_categorias)))
conteo_categorias.plot(kind='bar', color=colors, edgecolor='white')
plt.xlabel('Categoría')
plt.ylabel('Número de Noticias')
plt.title('Distribución de Noticias por Categoría', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nDistribución por categoría:")
print(conteo_categorias)

## 14. Sentimiento por Categoría

In [ ]:
# Calcular sentimiento promedio por categoría
sentimiento_por_categoria = df_noticias.groupby('categoria')['sentimiento'].agg(['mean', 'count'])
sentimiento_por_categoria = sentimiento_por_categoria.sort_values('mean', ascending=True)

if len(sentimiento_por_categoria) > 0:
    plt.figure(figsize=(10, 6))
    colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in sentimiento_por_categoria['mean']]
    plt.barh(sentimiento_por_categoria.index, sentimiento_por_categoria['mean'], color=colors)
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.xlabel('Sentimiento Promedio')
    plt.ylabel('Categoría')
    plt.title('Sentimiento Promedio por Categoría', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\nSentimiento promedio por categoría:")
print(sentimiento_por_categoria)

## 15. Exportar Resultados

In [ ]:
# Seleccionar columnas para exportar
columnas_exportar = ['titulo', 'descripcion_limpia', 'fecha', 'fuente', 
                     'sentimiento', 'clasificacion_sentimiento', 'categoria', 'link']

# Filtrar columnas que existen
columnas_disponibles = [c for c in columnas_exportar if c in df_noticias.columns]
df_exportar = df_noticias[columnas_disponibles]

# Exportar a CSV
nombre_archivo = 'yahoo_finance_noticias_analisis.csv'
df_exportar.to_csv(nombre_archivo, index=False, encoding='utf-8')

print(f"Datos exportados a '{nombre_archivo}'")
print(f"Total de filas exportadas: {len(df_exportar)}")

## 16. Resumen Ejecutivo

In [ ]:
print("="*70)
print("                    RESUMEN EJECUTIVO")
print("="*70)
print(f"\nFecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nTotal de noticias analizadas: {len(df_noticias)}")

print("\n--- SENTIMIENTO GENERAL ---")
sentimiento_counts = df_noticias['clasificacion_sentimiento'].value_counts()
for sent, count in sentimiento_counts.items():
    pct = (count / len(df_noticias)) * 100
    print(f"  {sent}: {count} noticias ({pct:.1f}%)")

print(f"\n  Polaridad promedio: {df_noticias['sentimiento'].mean():.3f}")

# Determinar el tono general
polaridad_promedio = df_noticias['sentimiento'].mean()
if polaridad_promedio > 0.1:
    tono = "POSITIVO"
elif polaridad_promedio < -0.1:
    tono = "NEGATIVO"
else:
    tono = "NEUTRAL"
print(f"  Tono general del mercado: {tono}")

print("\n--- CATEGORÍAS PRINCIPALES ---")
top_cats = df_noticias['categoria'].value_counts().head(3)
for cat, count in top_cats.items():
    print(f"  {cat}: {count} noticias")

print("\n--- PALABRAS CLAVE TOP 10 ---")
for i, (palabra, freq) in enumerate(conteo_palabras.most_common(10), 1):
    print(f"  {i}. {palabra}: {freq} menciones")

print("\n" + "="*70)